# 多変数線形回帰モデルによる異常検知

7.3節では説明変数が1つの線形回帰を紹介しましたが、一般的には複数の説明変数を組み合わせた方が応答変数をより正確にモデル化できます。
例えば7.3節の1変数`year`のみを説明変数とした実装では、多くの見逃しが発生していました。これに対し、説明変数に`odometer`（総走行距離）、`model_name`（車種）も加えることで、応答変数`price`の平均値やばらつきをより高精度に推定できます。これにより、より多角的にお買い得な中古車を
絞り込めるようになります。

ここでは多変数線形回帰モデルによる異常検知を、Pythonを用いて以下の手順で実装する方法を解説します。

- A. 説明変数の選択
- B. モデルの学習
- C. 推論

なおデータセットには7.2節で作成したサンプルデータを使用し、説明変数の候補として`year`（経過年数）、`odometer`（総走行距離）、`model_name`（車種。この変数のみカテゴリ変数）の3変数を、ターゲットとする誤報率としては0.0027を採用します。

## A. 説明変数の選択

ホテリング理論の変数選択と同様、入出力があるデータの異常検知においても、使用できる説明変数を全てモデルに入力することが最良とは限りません。多数の説明変数をモデルに入力すると、多重共線性や次元の呪い、説明可能性の低下等を引き起こす可能性があります。よって一般的には適切な変数を根拠を持って選択することが、モデルの性能や説明可能性、ロバスト性の向上につながります。

入出力があるデータの変数選択においても、主に以下の2種類の方法が利用できます。

- 7.2節の可視化に基づく定性的な変数選択
- 9章「変数選択」で紹介する各種手法に基づく、より定量的な変数選択

後者の方法は9章であらためて解説するため、ここでは簡易的な方法として、7.2節の可視化手法のうち次に示す方法を用いることとします。

- 箱ひげ図によるカテゴリ変数の寄与の確認
- 散布図と単回帰直線による目的変数との相関の確認
- 散布図による正常と異常の分離性の確認（異常データが取得できる場合のみ）

### 箱ひげ図によるカテゴリ変数の寄与の確認

箱ひげ図による可視化を通じて、カテゴリ変数`model_name`が応答変数`price`の変化に寄与するかを確認します。

In [ ]:
# 箱ひげ図によるカテゴリ変数の寄与の確認

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# CSVからPandas DataFrameにデータ読み込み
df = pd.read_csv('./datasets/usedcar_dataset_train.csv')
# 正常データのみ使用
df_normal = df[df['label'] == 'normal']

# boxplotによる箱ひげ図の描画
sns.boxplot(
    data=df_normal, # DataFrame
    x='model_name', # 横軸とするカテゴリ説明変数名
    y='price', # 縦軸とする応答変数名
    color='#999999', # 塗りつぶし色
)
plt.show()

カテゴリ変数`model_name`（車種）によって、明確に応答変数が変化しているように見えるため、`model_name`は説明変数に加えた方が良さそうです。後ほど他の説明変数との関係性も可視化した上で、総合的に判断していきます。

### 散布図と単回帰直線による目的変数との相関の確認

散布図と単回帰直線による可視化で、各説明変数候補と目的変数`price`との相関関係を確認し、相関があるものを説明変数として採用していきます。

In [ ]:
# 散布図と単回帰直線による目的変数との相関の確認
# 描画用のFigureとAxesを生成
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
# regplotによる単回帰直線と散布図の描画
for i, colname in enumerate(['year', 'odometer']):
    sns.regplot(
        data=df_normal, # DataFrame
        x=colname, # 横軸とする説明変数名
        y='price', # 縦軸とする応答変数名
        scatter_kws={'color':'#999999'}, # 点の色
        line_kws={'color':'#111111'}, # 線の色
        ci=None, # 回帰直線の信頼区間の表示の有無
        ax=axes[i] # 描画対象のAxes
        )
    

`odometer`と`price`は単回帰直線を見ると一見相関がないように見えますが、散布図の分布が複数のクラスタに分かれており、クラスタ内では左上から右下にかけて相関関係があるようにも見えます。カテゴリ変数`model_name`がこのクラスタに寄与していないかを確認するため、`model_name`ごとに分けて散布図と単回帰直線を再度プロットしてみます。

In [ ]:
# `model_name`（車種）ごとに分けて散布図と単回帰直線の描画
# model_nameごとにデータを分割してループ
for j, (model_name, data_model) in enumerate(df_normal.groupby('model_name')):
    # 描画用のFigureとAxesを生成
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
    # regplotによる単回帰直線と散布図の描画
    for i, colname in enumerate(['year', 'odometer']):
        sns.regplot(
            data=data_model, # DataFrame
            x=colname, # 横軸とする説明変数名
            y='price', # 縦軸とする応答変数名
            scatter_kws={'color':'#999999'}, # 点の色
            line_kws={'color':'#111111'}, # 線の色
            ci=None, # 回帰直線の信頼区間の表示の有無
            ax=axes[i], # 描画対象のAxes
            )
        # x軸、y軸範囲を統一
        axes[i].set_xlim(df_normal[colname].min(), df_normal[colname].max())
        axes[i].set_ylim(df_normal['price'].min(), df_normal['price'].max())
    plt.suptitle(f'model_name: {model_name}')
    plt.show()

`model_name`（車種）ごとにプロットを分けることで、`odometer`と応答変数`price`との相関関係が明確になりました。このことから、`year`、`odometer`どちらも説明変数に加えた方が良さそうです。

### 散布図による正常と異常の分離性の確認

この方法は異常データが取得できる場合に限られますが、散布図を用いて正常と異常の分離性も確認してみます。まずは`model_name`（車種）を分けずにプロットしてみます。

In [ ]:
# 散布図による正常と異常の分離性の確認
# 描画用のFigureとAxesを生成
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
# regplotによる単回帰直線と散布図の描画
for i, colname in enumerate(['year', 'odometer']):
    sns.scatterplot(
        data=df, # DataFrame
        x=colname, # 横軸とする説明変数名
        y='price', # 縦軸とする応答変数名
        hue='label', # 正常と異常のラベルで色分け
        palette=["#999999", "#111111"], # 色分けのカラーパレット
        ax=axes[i] # 描画対象のAxes
        )

正常と異常がうまく分離していなさそうです。`model_name`（車種）を分けて再度プロットします。

In [ ]:
# `model_name`（車種）ごとに分けて散布図による正常と異常の分離性を確認
# model_nameごとにデータを分割してループ
for j, (model_name, data_model) in enumerate(df.groupby('model_name')):
    # 描画用のFigureとAxesを生成
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
    # regplotによる単回帰直線と散布図の描画
    for i, colname in enumerate(['year', 'odometer']):
        sns.scatterplot(
            data=data_model, # DataFrame
            x=colname, # 横軸とする説明変数名
            y='price', # 縦軸とする応答変数名
            hue='label', # 正常と異常のラベルで色分け
            palette=["#999999", "#111111"], # 色分けのカラーパレット
            ax=axes[i] # 描画対象のAxes
        )
        # x軸、y軸範囲を統一
        axes[i].set_xlim(df_normal[colname].min(), df_normal[colname].max())
        axes[i].set_ylim(df_normal['price'].min(), df_normal['price'].max())
    plt.suptitle(f'model_name: {model_name}')
    plt.show()

正常と異常がうまく分離していそうです。このことから、`year`（経過年数）、`odometer`（総走行距離）、`model_name`（車種）の3変数全てを説明変数として用いることで、ある程度の性能の異常検知の実現が期待できます。

## B. モデルの学習

学習フェーズでは、最尤推定による線形回帰モデルのパラメータ推定、および分位点に基づく異常度のしきい値を算出します。1変数の場合と同様、statsmodelsの
[statsmodels.regression.linear_model.OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html)クラスを使用して線形回帰モデルを構築します。

なお車種を表す説明変数`model_name`はカテゴリ変数のため、前処理としてone-hot encodingによるカテゴリ変数の数値化（9章を参照）を実施します。

In [ ]:
# コード7.9 線形回帰による多説明変数の異常検知の実装例（学習）
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm

###### 学習データの読み込みと前処理######
# CSVからPandas DataFrameに学習データ読み込み
df = pd.read_csv('./datasets/usedcar_dataset_train.csv')
# 正常データのみを抽出
df_normal = df[df['label'] == 'normal']
# カテゴリ変数'model_name'の数値化（one-hot encoding）
enc = OneHotEncoder(drop='first')
model_name_cat = enc.fit_transform(df_normal[['model_name']].to_numpy()).toarray()
model_cat_names = [f'model_name_{col}' for col in enc.categories_[0][1:]]
df_model_name = pd.DataFrame(model_name_cat, columns=model_cat_names)
df_encoded = pd.concat([df_normal.drop('model_name', axis=1), df_model_name], axis=1)
# 学習データの説明変数と応答変数（'price'）を別々に保持（応答変数のみndarray化）
X_train = df_encoded[['year', 'odometer']+model_cat_names]
y_train = df_encoded['price'].to_numpy()

###### 学習ステップ1. 正常のモデルを作成する######
X_train_intercept = sm.add_constant(X_train) # 切片を追加
mod = sm.OLS(y_train, X_train_intercept) # 線形回帰モデル（OLS）を作成
res = mod.fit() # モデルの学習を実行
w = res.params[1:].to_numpy() # パラメータw
w_0 = res.params[0] # パラメータw_0
sigma2 = res.scale*res.df_resid / mod.nobs # パラメータσ^2（scaleは不偏推定値）

###### 学習ステップ2. 異常を表す指標（異常度）を定義する######
# 式を定義するのみでプログラム上は処理を実施しない

###### 学習ステップ3. 異常度にしきい値を設ける######
TARGET_FP_RATE = 0.0027 # ターゲットとする誤報率（正規分布の3σ相当=0.0027）
# 異常度を求める
x_train_anom_score = (y_train-w@X_train.T.to_numpy()-w_0)**2 / sigma2
# 異常度の分位点からしきい値算出
a_th = np.quantile(x_train_anom_score, 1-TARGET_FP_RATE)

###### 学習で求めたパラメータをすべて表示######
print(f'w={w}')
print(f'w_0={w_0}')
print(f'sigma2={sigma2}')
print(f'a_th={a_th}')

推定されたパラメータから求めた確率密度関数$p(y \mid x, \hat{\boldsymbol{w}}, \hat{w}_0, \hat{\sigma}^2)$と学習データを重ねてプロットしてみます。

In [ ]:
# 学習した線形回帰モデルの確率密度関数と学習データを重ねてプロット
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats

# 描画用のFigureとAxesを生成
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))
# 削除したカテゴリの列を表示用に追加する
droped_model_name = enc.categories_[0][0]
X_train_with_dropped = X_train.copy()
X_train_with_dropped[f'model_name_{droped_model_name}'] = (df_normal['model_name'] == droped_model_name).astype('float64')
# 車種ごとに図を分ける
for model_idx, model_cat_name in enumerate(model_cat_names+[f'model_name_{droped_model_name}']):
    X_train_model = X_train[X_train_with_dropped[model_cat_name] == 1]
    y_train_model = y_train[X_train_with_dropped[model_cat_name] == 1]
    # yearごとに図を分ける
    for year_idx, year in enumerate([1, 5, 10]):
        X_train_year = X_train_model[X_train_model['year'] == year]
        y_train_year = y_train_model[X_train_model['year'] == year]
        ###### 確率密度関数を描画 ######
        # (x,y)格子点を作成
        x_min, x_max = np.min(X_train['odometer']), np.max(X_train['odometer'])
        y_min, y_max = np.min(y_train), np.max(y_train)
        x_grid = np.linspace(x_min-1, x_max+2, num=100)
        y_grid = np.linspace(y_min-50, y_max+50, num=200)
        X, Y = np.meshgrid(x_grid, y_grid)
        XY_grid = np.c_[X.ravel(), Y.ravel()]
        # 格子点の説明変数データに車種とyearを追加
        X_all_grid = XY_grid[:, 0].reshape(-1, 1)
        for i_model in range(len(model_cat_names)):
            if i_model == model_idx:
                X_all_grid = np.insert(X_all_grid, i_model+1, 1, axis=1)
            else:
                X_all_grid = np.insert(X_all_grid, i_model+1, 0, axis=1)
        X_all_grid = np.insert(X_all_grid, 0, year, axis=1)  # year
        # 確率密度関数
        mu_all_grid = w@X_all_grid.T + w_0  # 線形予測子で平均μを予測
        pdf_all_grid = stats.norm.pdf(XY_grid[:, 1], loc=mu_all_grid, scale=np.sqrt(sigma2))
        # 確率密度をプロット
        pdf_grid_pivot = pdf_all_grid.reshape(X.shape)  # ピボット化
        axes[year_idx][model_idx].contourf(X, Y, pdf_grid_pivot, levels=5, cmap=cm.gray, alpha=0.5)

        ###### 学習データを散布図で描画 ######
        sns.scatterplot(x=X_train_year['odometer'], y=y_train_year,
                        c='#333333', ax=axes[year_idx][model_idx], s=18, marker="o")
        axes[year_idx][model_idx].set_xlabel('odometer', fontsize=12)
        axes[year_idx][model_idx].set_ylabel('price', fontsize=12)
        axes[year_idx][model_idx].set_title(f'{model_cat_name.replace("name_", "name=")},  year={year}', fontsize=14)
# グラフを表示
plt.tight_layout()
plt.show()


個々のグラフは横軸に説明変数`year`、縦軸に応答変数`price`をとっており、横方向には車種（`model_name`）、縦方向には経過年数（`year`= 1, 5, 10）を変えたときのグラフを並べています。パラメータの推定値やプロットされた図から、以下のことが確認できます。

- 1変数モデルでは経過年数（`year`）のみに応じて分布が変化していたが、多変数モデルでは他の説明変数`model_name`、`odometer`の影響も取り入れ、より精緻に分布が推定できている。
- 推定されたパラメータ（$\hat{w}_1=-10.668,\hat{w}_2=-3.801,\hat{w}_3[0]=48.584,\hat{w}_3[1]=145.578,\hat{w}_0=348.563,\hat{\sigma}^2=606.888$）は、データセット生成時における真値（$w_1=-10,w_2=-5,w_d[0]=50,w_d[1]=150,w_0=350,\sigma^2\simeq20^2+15^2=625$）と近く、高精度で学習できている（切片の基準カテゴリが`"sedan"`から`"compact"`に変わっている点に注意）
- 特に分散$\sigma^2$は1変数モデル（$\sigma^2=3583.752$）と比べ推定精度が大きく改善。これは1変数モデルでは考慮できなかった`model_name`や`odometer`の影響が分散に吸収されてしまっていたのに対し、多変数モデルでは平均パラメータ$\mu$の推定に正しく反映され、残差分散が適切に抑えられたため

3番目の例のように説明変数の不足は分散の増⼤につながることが多く、逆に言えば分布から大きく外れたデータを調べることで未知の説明変数を発見できることもあります。この点は異常検知アルゴリズムを活用する上で重要な視点となります。

なお学習後には、求めたパラメータやしきい値だけでなく、one-hot encoding用のモデルも以下のようにpickleで保存してください。

In [ ]:
# コード7.10 pickleによるone-hot encoding 用モデルの保存
import pickle
filepath = './7_4_3_onehot_encoder.pkl' # モデルの保存ファイル名
with open(filepath,'wb') as p: #ファイルを開く
    pickle.dump(enc, p) # pickleでファイルにモデルを保存する

## C. 推論

学習フェーズで保存したモデル（線形回帰モデル＆one-hot encoding用モデル）としきい値を用いて、推論データに対する異常度の算出と異常判定を行います。

まず以下のように、学習時に保存したone-hot encoding用のモデルを`enc`変数に読み込んでおきます。

In [ ]:
# コード7.11 pickleで保存したone-hot encoding 用モデルの読み込み
import pickle
filepath = './7_4_3_onehot_encoder.pkl' # モデルの保存ファイル名
with open(filepath,'rb') as p: #ファイルを開く
    enc = pickle.load(p) # pickle形式ファイルからモデルを読み込み

ここから以下の実装で、学習で求めたパラメータとしきい値を用いて線形回帰モデルによる異常判定を実施します。

In [ ]:
# コード7.12 線形回帰による多説明変数の異常検知の実装例（推論）
###### 学習したパラメータをここに記載######
# 線形予測子の係数パラメータ
W = [-10.66841099, -3.80082644, 48.58413929, 145.57788034]
# 線形予測子の切片パラメータ
W_0=348.5625021021391
# 残差の分散パラメータσ^2
SIGMA2=606.8877296523079
# 異常度のしきい値
A_TH=7.633084668394211

###### 推論データの読み込みと前処理######
# CSVからPandas DataFrameにデータ読み込み
df_inference = pd.read_csv('./datasets/usedcar_dataset_inference.csv')
# カテゴリ変数'model_name'の数値化（one-hot encoding）
model_name_cat = enc.transform(df_inference[['model_name']].to_numpy()).toarray()
model_cat_names = [f'model_name_{col}' for col in enc.categories_[0][1:]]
df_model_name = pd.DataFrame(model_name_cat, columns=model_cat_names)
df_encoded = pd.concat([df_inference.drop('model_name', axis=1), df_model_name], axis=1)
# 学習データの説明変数（'year'）と応答変数（'price'）をNumpyのndarray化
X_inference = df_encoded[['year', 'odometer']+model_cat_names].to_numpy()
y_inference = df_encoded['price'].to_numpy()

###### 推論を実行######
# 異常度を算出
anomaly_scores = (y_inference-W@X_inference.T-W_0)**2 / SIGMA2
# しきい値により異常の有無を判定
pred = np.where(anomaly_scores > A_TH, 'anomaly', 'normal')
# 推論結果を表示
print(pred)

推論結果の決定境界（異常と正常の判定の境界）を可視化してみます。個々のグラフは横軸に説明変数`year`、縦軸に応答変数`price`をとっており、横方向には車種（`model_name`）、縦方向には経過年数（`year`= 1, 5, 10）を変えたときのグラフを並べています。

In [ ]:
# 推論結果の決定境界を可視化
# 描画用のFigureとAxesを生成
fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(15, 12))

###### 正常と異常の範囲を色分け ######
# 削除したカテゴリの列を表示用に追加する
droped_model_name = enc.categories_[0][0]
df_X_inference = df_encoded[['year', 'odometer'] + model_cat_names]
X_inference_with_dropped = df_X_inference.copy()
X_inference_with_dropped[f'model_name_{droped_model_name}'] = (df_inference['model_name'] == droped_model_name).astype('float64')
# 車種ごとに図を分ける
for model_idx, model_cat_name in enumerate(model_cat_names+[f'model_name_{droped_model_name}']):
    df_inference_model = df_inference[X_inference_with_dropped[model_cat_name] == 1]
    # yearごとに図を分ける
    for year_idx, year in enumerate([1, 5, 10]):
        df_inference_year = df_inference_model[df_inference_model['year'] == year]
        ###### 確率密度関数を描画 ######
        # (x,y)格子点を作成
        x_min, x_max = np.min(df_X_inference['odometer']), np.max(df_X_inference['odometer'])
        y_min, y_max = np.min(y_inference), np.max(y_inference)
        x_grid = np.linspace(x_min-1, x_max+2, num=100)
        y_grid = np.linspace(y_min-50, y_max+50, num=200)
        X, Y = np.meshgrid(x_grid, y_grid)
        XY_grid = np.c_[X.ravel(), Y.ravel()]
        # 格子点の説明変数データに車種とyearを追加
        X_all_grid = XY_grid[:, 0].reshape(-1, 1)
        for i_model in range(len(model_cat_names)):
            if i_model == model_idx:
                X_all_grid = np.insert(X_all_grid, i_model+1, 1, axis=1)
            else:
                X_all_grid = np.insert(X_all_grid, i_model+1, 0, axis=1)
        X_all_grid = np.insert(X_all_grid, 0, year, axis=1)  # year
        # 異常度を算出
        anomaly_scores_grid = (XY_grid[:, 1]-W@X_all_grid.T-W_0)**2 / SIGMA2
        # しきい値判定
        pred_grid = np.where(anomaly_scores_grid > A_TH, 0, 1)
        # 正常と異常の境界をプロット
        pred_pivot = pred_grid.reshape(X.shape)
        axes[year_idx][model_idx].contourf(X, Y, pred_pivot, levels=1,
                            cmap=cm.gray, alpha=0.5)

        ###### 各データを散布図としてプロット ######
        sns.scatterplot(data=df_inference_year, x='odometer', y='price',
                        hue='label', palette=['#999999', '#111111'], ax=axes[year_idx][model_idx])
        # 図タイトルと凡例を追加
        axes[year_idx][model_idx].set_title(f'{model_cat_name.replace("name_", "name=")},  year={year}', fontsize=14)
        axes[year_idx][model_idx].legend()
# グラフを表示
plt.tight_layout()
plt.show()

1説明変数の場合（`year`のみを使用）と比べると、説明変数として車種（`model_name`）や（`odometer`）を追加したことで、その変化に追従して正常範囲を変化させ、より多くの異常データを正しく検知できていることがわかります。

ただし注意すべき点もあります。応答変数に変化を与える適切な説明変数をモデルに加えると一般的に性能が上がりますが、無関係な説明変数を加えると逆に性能が下がる場合もあります。実務ではドメイン知識や、9章で紹介する変数選択の手法を活用して、適切な説明変数を事前に見極めることが重要です。